In [8]:
# Conservative venue merging - only merge clear duplicates with location suffixes
import sqlite3
import pandas as pd
from pathlib import Path
from fuzzywuzzy import fuzz

# Connect to database
project_root = Path().resolve().parent if Path().resolve().name == 'research' else Path().resolve()
db_path = str(project_root / 't20s_json' / 'venue_metrics.db')
conn = sqlite3.connect(db_path)

# Get all unique venues from first_innings_sections
venues_df = pd.read_sql_query("SELECT DISTINCT venue FROM first_innings_sections ORDER BY venue", conn)
print(f"Found {len(venues_df)} unique venues in first_innings_sections table")

def conservative_merge_venues(venues_list):
    """
    Very conservative venue merging:
    - Only merges if one name is EXACTLY the prefix of another + comma + location
    - Example: "M Chinnaswamy Stadium" merges with "M Chinnaswamy Stadium, Bengaluru"
    - Uses high token_sort_ratio (90+) for minor variations like punctuation
    """
    merged_mapping = {}
    
    # Sort by length so we process shorter names first
    sorted_venues = sorted(venues_list, key=len)
    
    for venue in sorted_venues:
        if venue in merged_mapping:
            continue  # Already mapped
            
        # This venue maps to itself initially
        canonical = venue
        merged_mapping[venue] = canonical
        
        # Check if this venue is a prefix of any longer venue
        venue_lower = venue.lower().strip()
        
        for other_venue in venues_list:
            if other_venue == venue or other_venue in merged_mapping:
                continue
                
            other_lower = other_venue.lower().strip()
            
            # Check for exact prefix match followed by comma
            # e.g., "M Chinnaswamy Stadium" vs "M Chinnaswamy Stadium, Bengaluru"
            if other_lower.startswith(venue_lower):
                remainder = other_lower[len(venue_lower):].strip()
                # Only merge if the remainder starts with comma (location suffix)
                if remainder.startswith(',') or remainder.startswith(' ,'):
                    merged_mapping[other_venue] = canonical
                    continue
            
            # Also check reverse - if venue is longer
            if venue_lower.startswith(other_lower):
                remainder = venue_lower[len(other_lower):].strip()
                if remainder.startswith(',') or remainder.startswith(' ,'):
                    # The shorter one (other_venue) should be canonical
                    if other_venue not in merged_mapping:
                        merged_mapping[other_venue] = other_venue
                    merged_mapping[venue] = other_venue
                    canonical = other_venue
                    break
            
            # Use high threshold token_sort_ratio only for very close matches
            # This catches punctuation differences like "M.Chinnaswamy" vs "M Chinnaswamy"
            token_score = fuzz.token_sort_ratio(venue_lower, other_lower)
            if token_score >= 95:  # Very high threshold - almost identical
                # Choose shorter as canonical
                if len(other_venue) < len(venue):
                    if other_venue not in merged_mapping:
                        merged_mapping[other_venue] = other_venue
                    merged_mapping[venue] = other_venue
                    canonical = other_venue
                    break
                else:
                    merged_mapping[other_venue] = canonical
    
    return merged_mapping

# Create conservative venue mapping
venues_list = venues_df['venue'].tolist()
venue_mapping = conservative_merge_venues(venues_list)

# Show merging stats
merged_count = sum(1 for orig, clean in venue_mapping.items() if orig != clean)
unique_clean_venues = len(set(venue_mapping.values()))
print(f"\nConservative venue merging results:")
print(f"  Original unique venues: {len(venues_list)}")
print(f"  After merging: {unique_clean_venues}")
print(f"  Venues merged: {merged_count}")

# Update venue_mapping table in database
venue_mapping_df = pd.DataFrame([
    {'venue_original': orig, 'venue_clean': clean}
    for orig, clean in venue_mapping.items()
])

venue_mapping_df.to_sql('venue_mapping', conn, if_exists='replace', index=False)
conn.execute("CREATE INDEX IF NOT EXISTS idx_venue_mapping_original ON venue_mapping(venue_original)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_venue_mapping_clean ON venue_mapping(venue_clean)")

print(f"\nUpdated 'venue_mapping' table with {len(venue_mapping_df)} rows")

# Recreate first_innings_sections_clean table
print("\nRecreating 'first_innings_sections_clean' table...")

conn.execute("DROP TABLE IF EXISTS first_innings_sections_clean")

query_create = """
CREATE TABLE first_innings_sections_clean AS
SELECT 
    fis.match_id,
    fis.date,
    fis.team,
    COALESCE(vm.venue_clean, fis.venue) as venue,
    fis.section,
    fis.overs,
    fis.overs_completed,
    fis.runs_in_section,
    fis.wickets_in_section,
    fis.section_run_rate,
    fis.cumulative_runs,
    fis.cumulative_wickets,
    fis.cumulative_run_rate,
    fis.powerplay_completed,
    fis.balls_remaining,
    fis.fours_hit,
    fis.sixes_hit,
    fis.boundaries_last_2_overs,
    fis.dot_balls_so_far,
    fis.dot_balls_last_2_overs
FROM first_innings_sections fis
LEFT JOIN venue_mapping vm ON fis.venue = vm.venue_original
"""

conn.execute(query_create)

# Create indexes
conn.execute("CREATE INDEX IF NOT EXISTS idx_fis_clean_date ON first_innings_sections_clean(date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fis_clean_venue ON first_innings_sections_clean(venue)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fis_clean_team ON first_innings_sections_clean(team)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fis_clean_overs ON first_innings_sections_clean(overs_completed)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fis_clean_match_section ON first_innings_sections_clean(match_id, section)")

conn.commit()

# Verify
row_count = conn.execute("SELECT COUNT(*) FROM first_innings_sections_clean").fetchone()[0]
venue_count = conn.execute("SELECT COUNT(DISTINCT venue) FROM first_innings_sections_clean").fetchone()[0]

print(f"✓ Recreated 'first_innings_sections_clean' table")
print(f"  Total rows: {row_count}")
print(f"  Unique venues (after merging): {venue_count}")

# Show sample merged venues
print("\nVenues that were merged (showing all):")
merged_venues = {}
for orig, clean in venue_mapping.items():
    if orig != clean:
        if clean not in merged_venues:
            merged_venues[clean] = []
        merged_venues[clean].append(orig)

if merged_venues:
    for i, (canonical, variants) in enumerate(sorted(merged_venues.items()), 1):
        print(f"\n{i}. '{canonical}' ← merged with:")
        for variant in sorted(variants):
            print(f"   • {variant}")
else:
    print("  No venues were merged")

# Verify no incorrect merges
print("\n" + "="*70)
print("Checking for incorrect merges:")
print("="*70)

# Check that different stadiums are NOT merged together
different_stadiums = [
    'Amini Park',
    'Bermuda National Stadium', 
    'Melbourne Cricket Ground',
    'Wankhede Stadium',
    'AMI Stadium',
    'Barabati Stadium',
    'Gaddafi Stadium'
]

canonical_names = set()
for stadium in different_stadiums:
    matches = [v for v in venues_list if stadium.lower() in v.lower()]
    for v in matches:
        canonical_names.add(venue_mapping.get(v, v))

print(f"\nFound {len(canonical_names)} different canonical names for test stadiums")
print("(Should be 7+ if stadiums are correctly kept separate)")

# Show specific examples
print("\nExamples:")
test_examples = ['M Chinnaswamy Stadium', 'M Chinnaswamy Stadium, Bengaluru', 
                 'M.Chinnaswamy Stadium', 'Wankhede Stadium, Mumbai',
                 'Melbourne Cricket Ground']
for example in test_examples:
    if example in venue_mapping:
        print(f"  '{example}' → '{venue_mapping[example]}'")

conn.close()
print("\n✓ Done!")

Found 322 unique venues in first_innings_sections table

Conservative venue merging results:
  Original unique venues: 322
  After merging: 236
  Venues merged: 86

Updated 'venue_mapping' table with 322 rows

Recreating 'first_innings_sections_clean' table...
✓ Recreated 'first_innings_sections_clean' table
  Total rows: 28954
  Unique venues (after merging): 236

Venues that were merged (showing all):

1. 'Achimota Senior Secondary School A Field, Accra' ← merged with:
   • Achimota Senior Secondary School B Field, Accra

2. 'Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)' ← merged with:
   • Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)

3. 'Amini Park' ← merged with:
   • Amini Park, Port Moresby

4. 'Arnos Vale Ground, Kingstown' ← merged with:
   • Arnos Vale Ground, Kingstown, St Vincent

5. 'Arun Jaitley Stadium' ← merged with:
   • Arun Jaitley Stadium, Delhi

6. 'Barabati Stadium' ← merged with:
   • Barabati Stadium, Cuttack

7. 'Barsapara Cricket Stadium'